**Feature Selection Method:** ANOVA   
**Dataset:** QUT-DV25 (TCP)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif

In [ ]:

# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [ ]:
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [ ]:
data.head()

,Package_Name,Total_Entries,Unique_C-COMM,Python_Related_Process,State_Transition,Local_IP_Address_Access,Remote_IP_Address_Access,Local_Port_Access,Remote_Port_Access,Level
0,10Cent10-999.0.4.tar.gz,946,7,316,"{'FIN_WAIT1 -> ->': 188, 'SYN_SENT -> ->': 187...",2,62,194,21,1
1,10Cent11-999.0.4.tar.gz,874,6,301,"{'SYN_SENT -> ->': 177, 'FIN_WAIT1 -> ->': 170...",2,51,181,18,1
2,11Cent-999.0.0.tar.gz,801,4,272,"{'FIN_WAIT1 -> ->': 160, 'SYN_SENT -> ->': 157...",2,52,164,17,1
3,11Cent-999.0.1.tar.gz,728,4,254,"{'SYN_SENT -> ->': 149, 'FIN_WAIT1 -> ->': 141...",2,39,152,14,1
4,11Cent-999.0.2.tar.gz,655,4,234,"{'SYN_SENT -> ->': 133, 'FIN_WAIT1 -> ->': 128...",2,32,135,11,1


In [ ]:
gc.collect()
tf.keras.backend.clear_session()

In [ ]:

# Separate target
X = data.drop(columns=['Level'])
y = data['Level']

if y.dtype == 'object' or y.dtype.name == 'category':
    y = LabelEncoder().fit_transform(y)

# Separate numerical and categorical features
numerical_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object','category']).columns.tolist()

X_num = X[numerical_cols]

# Encode categorical features as numeric codes
X_cat = X[categorical_cols].apply(lambda col: LabelEncoder().fit_transform(col.astype(str)))

# Combine numerical + encoded categorical features
X_encoded = pd.concat([X_num, X_cat], axis=1)

# Compute ANOVA F-scores
f_scores, _ = f_classif(X_encoded, y)

feature_df = pd.DataFrame({
    'feature': X_encoded.columns,
    'f_score': f_scores
})

# Select top 30% features by F-score
threshold = np.percentile(f_scores, 70)  # top 30%
feature_df['status'] = np.where(feature_df['f_score'] >= threshold, 'selected', 'deleted')

selected_features = feature_df[feature_df['status']=='selected']
deleted_features = feature_df[feature_df['status']=='deleted']

print("Selected features (name + F-score):\n", selected_features)
print("\nDeleted features (name + F-score):\n", deleted_features)


Selected features (name + F-score):
                     feature      f_score    status
0             Total_Entries   617.801279  selected
4  Remote_IP_Address_Access   488.592645  selected
6        Remote_Port_Access  1002.227790  selected

Deleted features (name + F-score):
                    feature     f_score   status
1            Unique_C-COMM   57.981335  deleted
2   Python_Related_Process   86.495256  deleted
3  Local_IP_Address_Access   85.724500  deleted
5        Local_Port_Access  431.261407  deleted
7             Package_Name   22.678863  deleted
8         State_Transition   53.817269  deleted
